# Programming for Data Science 
### Project Notebook: "Where should I live?" 
#### Group Members:
- Afonso Fernandes / 20241710
- Lourenço Lima / 20241711
- Pedro Jorge / 20241819
- David Morais / 20241759
## Project Repository
GitHub Repository:  
https://github.com/afonsolince06/-Where-should-I-live-PDS-Project


## **1. Introduction:**

While our initial dataset provides rich socio-economic indicators, it lacks the **geospatial coordinates** (Latitude and Longitude) required to build an interactive mapping tool. To bridge this gap, this notebook focuses on **Data Enrichment** through automated data collection.

Our goal is to fetch precise geographical data directly from Wikipedia for each city in our dataset, ensuring our final "Where Should I Live?" tool is both accurate and visually engaging.

### **Objectives of this Phase**
1.  **Automated Scraping:** Utilize **Selenium** to navigate Wikipedia pages and extract coordinate metadata for over 80 European cities.
2.  **Coordinate Transformation:** Convert coordinates from the DMS (Degrees, Minutes, Seconds) format typically found on Wikipedia to **Decimal Degrees** for compatibility with mapping libraries.
3.  **Data Integration:** Merge the newly acquired coordinates with our cleaned dataset (`city_data_cleaned.csv`).
4.  **Interactive Visualization:** Develop a dynamic map using **Plotly**, where users can visualize population density, salaries, and cost of living through spatial distribution.

### **Technical Setup & Tools**

To perform web scraping in a dynamic environment, we utilize the following stack:

- **Selenium WebDriver:** To simulate browser interaction and handle the search queries for each city.
- **BeautifulSoup:** For parsing the HTML content once the specific city page is loaded.
- **Plotly Express:** To render the interactive `scatter_map`, allowing for multi-layered data visualization (color for salary, size for population).

### **2. Importing the necessary libraries for webscraping and map building, and import the clean dataset.**

#### **2.1. Import essential libraries**

In [4]:
import pandas as pd
import numpy as np
import plotly.express as px
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

#### **2.2. Import the CSV File "city_data_cleaned.csv"**

In [5]:
city_data = pd.read_csv('city_data_cleaned.csv') # Load the cleaned city data
city_data_coords = city_data.copy() # Create a copy to add coordinates
# Fill the coordinate columns with NaN values
city_data_coords['Latitude'] = np.nan
city_data_coords['Longitude'] = np.nan

### **3. Web Scrapping**
In this section, Selenium is used to automatically retrieve the geographic coordinates (latitude and longitude) of European cities from the cleaned dataset `city_data_coords` using information available on Wikipedia. The process begins with the implementation of the function `get_city_coordinates`, which opens a web browser and navigates to the Wikipedia main page. For each city in the dataset, a search is performed—optionally including the country name to improve accuracy—followed by navigation to the corresponding city article, where the geographic coordinates are extracted.

The retrieved coordinates are subsequently stored in the dataset through the function `fill_dataset_coords`.

In [6]:
wikipedia_url = "https://en.wikipedia.org/wiki/Main_Page" # Wikipedia main page URL


def get_city_coordinates(city_name):
    """
    Starts on the Wikipedia main page and searches for the given city name.

    Parameters:
    city_name (str): The name of the city to search for.

    Returns:
    The city page URL.
    """

    if city_name == 'Rotterdam': # Special case for Rotterdam
        try:
            # Navigate to Wikipedia main page
            driver.get(wikipedia_url)

            # Find the search input box and enter the city name
            search_input = WebDriverWait(driver, 15).until(EC.element_to_be_clickable((By.ID, "searchInput")))
            search_input.send_keys(city_name)
            search_input.send_keys(Keys.RETURN)

        
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.ID, "firstHeading"))) # Wait for page to load
            lat = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//span[@class='latitude']")))
            latitude = driver.execute_script("return arguments[0].textContent;", lat)
            long = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//span[@class='longitude']")))
            longitude = driver.execute_script("return arguments[0].textContent;", long)
            return (latitude.strip(), longitude.strip())
        
        finally:
            driver.quit() # Close the browser

    else: # General case for other cities
        # Set up the Selenium WebDriver 
        options = webdriver.ChromeOptions()
        driver = webdriver.Chrome(options=options)
        driver.maximize_window()
        
        try:
            # Navigate to Wikipedia main page
            driver.get(wikipedia_url)
            # Find the search input box and enter the city name
            search_input = WebDriverWait(driver, 15).until(EC.element_to_be_clickable((By.ID, "searchInput")))
            search_input.send_keys(city_name)
            search_input.send_keys(Keys.RETURN)
          
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.ID, "firstHeading"))) # Wait for page to load
            city_latitude = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "latitude")))
            city_longitude = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "longitude")))
            return (city_latitude.text, city_longitude.text)
            
        finally:
            driver.quit() # Close the browser
    


def fill_dataset_coords(df):
    """
    Fills the dataset with latitude and longitude coordinates for each city.
    Parameters:
    df (DataFrame): The DataFrame containing city data.
    Returns:
    The updated DataFrame with latitude and longitude columns filled.
    """

    for city in df['City']:
        try: # Attempt to get coordinates using "City, Country" format first
            df.loc[df['City'] == city, ['Latitude', 'Longitude']] = get_city_coordinates(f"{city}, {df.loc[city_data_coords['City'] == city, 'Country'].values[0]}")
        except Exception: # If that fails, try just the city name
            df.loc[df['City'] == city, ['Latitude', 'Longitude']] = get_city_coordinates(city)
    return df





# Filling the dataset with coordinates
fill_dataset_coords(city_data_coords)


display(city_data_coords.tail(40)) # Checking the last 40 rows to check if the function worked for Rotterdam as it was the only city we had issues with
city_data_coords.loc[city_data_coords['City'] == 'Rotterdam', ['Latitude', 'Longitude']] = ('51°55′N', '4°29′E') # As we were not able to scrapte the coordinates for Rotterdam, we will fill it manually just so we have it in the map
display(city_data_coords.loc[city_data_coords['City'] == 'Rotterdam', ['City','Latitude', 'Longitude']]) # Checking if Rotterdam coordinates were filled correctly

C:\Users\afons\AppData\Local\Temp\ipykernel_30208\4113309029.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '47°48′00″N' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df['City'] == city, ['Latitude', 'Longitude']] = get_city_coordinates(f"{city}, {df.loc[city_data_coords['City'] == city, 'Country'].values[0]}")
C:\Users\afons\AppData\Local\Temp\ipykernel_30208\4113309029.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '13°02′42″E' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df['City'] == city, ['Latitude', 'Longitude']] = get_city_coordinates(f"{city}, {df.loc[city_data_coords['City'] == city, 'Country'].values[0]}")


ElementNotInteractableException: Message: element not interactable
  (Session info: chrome=142.0.7444.176); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementnotinteractableexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff7cb67a235
	0x7ff7cb3d2630
	0x7ff7cb1614e5
	0x7ff7cb1b3a13
	0x7ff7cb1b186a
	0x7ff7cb1e2b0a
	0x7ff7cb1acc36
	0x7ff7cb20baba
	0x7ff7cb1ab0ed
	0x7ff7cb1abf63
	0x7ff7cb6a5d60
	0x7ff7cb69fe8a
	0x7ff7cb6c1005
	0x7ff7cb3ed71e
	0x7ff7cb3f4e1f
	0x7ff7cb3db7c4
	0x7ff7cb3db97f
	0x7ff7cb3c18e8
	0x7fff65a67374
	0x7fff66b9cc91


### **Exporting the final DataFrame with the scraped coordinates (in DMS format)**

In [ ]:
city_data_coords.to_csv('city_data_with_coords.csv', index=False) # Saving the dataset with coordinates to a CSV file

# Safety net in case we need to reload the DataFrame with coordinates after deleting them by accident
city_data_coords = pd.read_csv('city_data_with_coords.csv')

### **4. Interactive Map**

This section is where we build the interactive map with all the cities in the provided dataset and the hover information that was asked for in the guidelines.

Since the coordinates obtained from Wikipedia are provided in DMS (Degrees, Minutes, Seconds) format, an additional function, `dms_to_decimal`, is applied to convert them into decimal degrees. These processed coordinates are then used as input for the construction of an interactive map, allowing for a clear geographical representation of the analyzed cities.

In [ ]:
# Iterative Map Building and Visualization

def dms_to_decimal(dms_coords):
    """
    Converts the coordinates from the DMS (Degrees, Minutes, Seconds) format that they were scraped in to a decimal format to be used in the map visualization.
    
    Parameters:
    dms_coords (str): The coordinates in DMS format.
    
    Returns:
    The coordinates in decimal format.
    """
    dms_parts = re.split('[°′″NEW]', dms_coords) # Split the DMS string into its components and taking care of N, E, W, S characters
    degrees = float(dms_parts[0]) # Degrees part is always present and it's the first element
    minutes = float(dms_parts[1]) if len(dms_parts) > 1 and dms_parts[1] else 0 # Minutes part may be missing, it's the second element
    seconds = float(dms_parts[2]) if len(dms_parts) > 2 and dms_parts[2] else 0 # Seconds part may be missing, it's the third element

    decimal_coords = degrees + (minutes / 60) + (seconds / 3600) # Formula to convert DMS to decimal

    if 'S' in dms_coords or 'W' in dms_coords:
        decimal_coords = -decimal_coords # Southern and western hemispheres are negative

    return decimal_coords

city_data_coords['Latitude'] = city_data_coords['Latitude'].apply(dms_to_decimal)
city_data_coords['Longitude'] = city_data_coords['Longitude'].apply(dms_to_decimal)

# Drop rows with missing coordinates so the map visualization works properly
map_df = city_data_coords.dropna(subset=['Latitude', 'Longitude'])

# Create the scatter map
fig_map = px.scatter_map(
        map_df,
        lat="Latitude",
        lon="Longitude",
        hover_name="City",
        hover_data={
            "Latitude": False, "Longitude": False,
            "Country": True,
            "Population": ':,.0f', 
            "Average Monthly Salary": ':,.0f €', 
            "Average Cost of Living": ':,.0f €'  
        },
        color="Average Monthly Salary", 
        size="Population", 
        size_max=25,             
        color_continuous_scale="Turbo",
        zoom=3, 
        height=700,
        title="Iterative Map of Cities with Population, Average Monthly Salary, and Cost of Living"
    )

# Update layout for better visualization
fig_map.update_layout(mapbox_style="open-street-map")
fig_map.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig_map.show()